In [2]:
import pandas as pd

df_clean = pd.read_csv("../data/processed/gaid_clean.csv")
selected_metrics = pd.read_csv("../data/processed/selected_metrics.csv")

In [3]:
df_selected = df_clean.merge(
    selected_metrics,
    on="Metric",
    how="inner"
)

In [4]:
df_selected.head()

,Year,Country,ISO3,Metric,Value,Dataset,Source,Source_Category,Source_File,Source_Type,Source_Year,Pillar,Weight,Direction
0,1998,Algeria,DZA,Number Of AI Publications: All,10.0,Stanford AI Index - Research and Development,Stanford AI Index,Research and Development,1. Research and Development-2021_Publications_...,xlsx,2021,Innovation,0.2,Higher is better
1,1998,Argentina,ARG,Number Of AI Publications: All,15.0,Stanford AI Index - Research and Development,Stanford AI Index,Research and Development,1. Research and Development-2021_Publications_...,xlsx,2021,Innovation,0.2,Higher is better
2,1998,Australia,AUS,Number Of AI Publications: All,894.0,Stanford AI Index - Research and Development,Stanford AI Index,Research and Development,1. Research and Development-2021_Publications_...,xlsx,2021,Innovation,0.2,Higher is better
3,1998,Austria,AUT,Number Of AI Publications: All,35.0,Stanford AI Index - Research and Development,Stanford AI Index,Research and Development,1. Research and Development-2021_Publications_...,xlsx,2021,Innovation,0.2,Higher is better
4,1998,Azerbaijan,AZE,Number Of AI Publications: All,2.0,Stanford AI Index - Research and Development,Stanford AI Index,Research and Development,1. Research and Development-2021_Publications_...,xlsx,2021,Innovation,0.2,Higher is better


In [5]:
df_latest = (
    df_selected
    .sort_values("Year")
    .groupby(["Country", "ISO3", "Metric"], as_index=False)
    .tail(1)
)

In [6]:
#Normalización por métrica
df_latest["NormalizedValue"] = df_latest.groupby("Metric")["Value"].transform(
    lambda x: ((x - x.min()) / (x.max() - x.min())) * 100 if x.max() != x.min() else 100
)

In [7]:
#Calcular score por pilar
pillar_scores = (
    df_latest
    .groupby(["Country", "ISO3", "Pillar"], as_index=False)
    .agg(PillarScore=("NormalizedValue", "mean"))
)

In [8]:
#Pasar a formato ancho
ai_scores = pillar_scores.pivot_table(
    index=["Country", "ISO3"],
    columns="Pillar",
    values="PillarScore",
    aggfunc="mean"
).reset_index()

In [9]:
#Calcular el score total
pillar_cols = [col for col in ai_scores.columns if col not in ["Country", "ISO3"]]

ai_scores["AI_Readiness_Score"] = ai_scores[pillar_cols].mean(axis=1)

In [10]:
#Ranking
ai_scores["Global_Rank"] = ai_scores["AI_Readiness_Score"].rank(
    ascending=False,
    method="dense"
).astype(int)

ai_scores = ai_scores.sort_values("Global_Rank")

In [11]:
#Exportación
ai_scores.to_csv("../data/processed/ai_readiness_scores.csv", index=False)